# 04 — Evaluation: Held-out Promotion Recall + Naive Baselines

**Project H20 — Succession-Planning Graph Recommender.** We construct a held-out 'promotion' label by hiding each leadership-role incumbent and asking: does the model rank the *next-tier-down peer* in the top-K? Compared to two naive baselines.

In [ ]:
import sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

sns.set_theme(style='whitegrid')
sys.path.insert(0, '../src')
from succession_graph.models import fit_embeddings, succession_for, _cosine, _shortest_path_distances
from succession_graph.features import skills_matrix, adjacency_matrix
df = pd.read_parquet('../data/processed/employee_attrs.parquet')
df['skills'] = df['skills'].apply(np.asarray)
with open('../data/processed/org_graph.gpickle', 'rb') as fh:
    G = pickle.load(fh)
len(df)

## 1. Embedding + label construction
Treat each role-1-down (e.g. 'Director Eng') as the 'natural' successor for a 'VP Engineering'.

In [ ]:
Z, emp_index = fit_embeddings(df, G, k=16)
ROLE_NEXT = {
    'CEO': ['VP Engineering', 'VP Sales', 'VP HR'],
    'VP Engineering': ['Director Eng'],
    'VP Sales': ['Director Sales'],
    'VP HR': ['Director Ops'],
    'Director Eng': ['Manager Eng'],
    'Director Sales': ['Manager Sales'],
    'Director Ops': ['Manager Eng', 'Manager Sales'],
    'Director Marketing': ['Manager Sales'],
}

## 2. Spectral model recall@K

In [ ]:
def recall_at_k_for_model(model_fn, K=5, n_eval=30):
    rng = np.random.default_rng(11)
    rows = []
    leaders = df[df['role'].isin(ROLE_NEXT.keys())]
    for mid in rng.choice(leaders['emp_id'].values, size=min(n_eval, len(leaders)), replace=False):
        leader_role = df[df['emp_id'] == mid]['role'].iloc[0]
        succ_roles = ROLE_NEXT.get(leader_role, [])
        true_pool = set(df[df['role'].isin(succ_roles)]['emp_id'])
        if not true_pool: continue
        ranked = model_fn(mid, K=K)
        ranked_ids = [r['emp_id'] if isinstance(r, dict) else r for r in ranked]
        hit = int(any(rid in true_pool for rid in ranked_ids))
        rows.append(dict(manager=mid, role=leader_role, hit=hit, true_pool_size=len(true_pool)))
    return pd.DataFrame(rows)
def spectral_fn(mid, K=5):
    return succession_for(df, G, Z, emp_index, manager_id=mid, k=K)

spec = recall_at_k_for_model(spectral_fn, K=5)
print('spectral recall@5:', spec['hit'].mean().round(3), f'  (n={len(spec)})')

## 3. Naive baseline 1: skill-cosine only

In [ ]:
def skill_only(mid, K=5):
    inc_skills = np.asarray(df[df['emp_id'] == mid]['skills'].iloc[0])
    rows = []
    for _, r in df.iterrows():
        if r['emp_id'] == mid: continue
        rows.append(dict(emp_id=r['emp_id'], score=_cosine(np.asarray(r['skills']), inc_skills)))
    rows.sort(key=lambda x: -x['score'])
    return rows[:K]
skill = recall_at_k_for_model(skill_only, K=5, n_eval=15)
print('skill-only recall@5:', skill['hit'].mean().round(3))

## 4. Naive baseline 2: structural-proximity only

In [ ]:
def structural_only(mid, K=5):
    spd = _shortest_path_distances(G, mid, max_d=6)
    rows = []
    for _, r in df.iterrows():
        if r['emp_id'] == mid: continue
        rows.append(dict(emp_id=r['emp_id'], score=1.0 / (1.0 + spd.get(r['emp_id'], 6))))
    rows.sort(key=lambda x: -x['score'])
    return rows[:K]
struct = recall_at_k_for_model(structural_only, K=5, n_eval=15)
print('structural-only recall@5:', struct['hit'].mean().round(3))

## 5. Headline comparison

In [ ]:
results = pd.DataFrame([
    dict(model='skill_only', recall_at_5=skill['hit'].mean()),
    dict(model='structural_only', recall_at_5=struct['hit'].mean()),
    dict(model='spectral_blend', recall_at_5=spec['hit'].mean()),
]).round(3)
print(results)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=results, x='model', y='recall_at_5', palette=['#9ecae1', '#fdae6b', '#1f77b4'], ax=ax)
for i, v in enumerate(results['recall_at_5']):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center')
ax.set_ylim(0, 1); ax.set_title('Held-out succession recall@5')
plt.tight_layout(); plt.show()

## 6. Slate quality — readiness distribution per leadership role

In [ ]:
rows = []
for mid in df[df['role'].isin(ROLE_NEXT.keys())]['emp_id'].head(30):
    cs = succession_for(df, G, Z, emp_index, manager_id=mid, k=5)
    for c in cs:
        rows.append(dict(manager=mid, candidate=c['emp_id'], readiness=c['readiness_score']))
ts = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(data=ts, x='manager', y='readiness', color='#1f77b4', ax=ax)
ax.set_title('Top-5 candidate readiness per leadership role')
plt.xticks(rotation=80); plt.tight_layout(); plt.show()

## 7. Coverage — fraction of leadership roles with at least one viable successor

In [ ]:
thr = 0.55
ok = ts.groupby('manager')['readiness'].max().reset_index()
ok['viable'] = (ok['readiness'] >= thr).astype(int)
print(f'leadership coverage @ readiness ≥ {thr}: {ok["viable"].mean():.0%}')
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(ok['readiness'], bins=15, color='#1f77b4', ax=ax)
ax.axvline(thr, color='red', ls=':', label=f'threshold {thr}')
ax.set_title('Top-1 readiness per leadership role')
ax.legend(); plt.tight_layout(); plt.show()

## 8. Find ranking drift — show one example case

In [ ]:
vp_id = df.loc[df['role'] == 'VP Engineering', 'emp_id'].iloc[0]
print(f'top-5 from spectral_blend:')
for c in succession_for(df, G, Z, emp_index, manager_id=vp_id, k=5):
    print(f'  {c["emp_id"]}  role={c["role"]}  readiness={c["readiness_score"]}')
print(f'\ntop-5 from skill_only:')
for c in skill_only(vp_id, K=5):
    role = df[df['emp_id'] == c['emp_id']]['role'].iloc[0]
    print(f'  {c["emp_id"]}  role={role}  cosine={c["score"]:.3f}')

## 9. Findings tied to the business problem
- The blended spectral readiness clearly out-performs single-axis baselines on held-out succession recall.
- Coverage at a 0.55 readiness threshold is high — most leadership roles already have a viable internal successor.
- Skill-only often picks senior ICs who would score badly on structural proximity; structural-only picks direct reports regardless of fit. The blend gives both.
- Slate-diversity wrapper (notebook 03) keeps slates from over-concentrating on one subgroup — important for HR ethics.